In [2]:
""" ----- Task 1 og Task 4 ----- Character class og Specializations ----- """

from items import Weapon

class Character:
    """Represents a character in a game"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon: Weapon = None) -> None:
        """
        Initialize character:
        Args:
            name (str): The name of the character
            hp_max (int): Max HitPoints, default = 100
            damage (int): Damage dealt when a character attacks, default = 10
            weapon (Weapon): Optional weapon equipped by character
        """
        self.name = name
        self.hp_max = hp_max
        self.damage = damage
        self._hp = hp_max  # Beskyttet attributt for HP[cite: 2, 4]
        self.weapon = weapon

    # --- TASK 1 METODER ---
    def take_damage(self, amount: int) -> None:
        """Reduces the characters HP without going below 0"""
        self._hp = max(self._hp - amount, 0)

    def heal(self, amount: int) -> None:
        """Increases character health by amount without exceeding max health"""
        self._hp = min(self._hp + amount, self.hp_max)

    def attack(self, target) -> None:
        """Attack and deal damage to target character using weapon or base damage[cite: 2, 3]"""
        if self.weapon is not None:
            inflicted_damage = self.weapon.roll_damage()
        else:
            inflicted_damage = self.damage
        target.take_damage(inflicted_damage)

    def introduce(self) -> None:
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        return f"<Character name={self.name}, hp={self.hp}, damage={self.damage}, weapon={self.weapon}>"

    def __str__(self) -> str:
        weapon_str = f" with {self.weapon}" if self.weapon else ""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage {weapon_str}"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits[cite: 4]."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        """Public read-only property returning True if character is alive, else false"""
        return self._hp > 0


# --- TASK 4 SUBKLASSER (SPESIALISERINGER) ---

class Warrior(Character):
    """Represents a defensive specialist Warrior character"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon: Weapon = None, armor: int = 5) -> None:
        super().__init__(name, hp_max, damage, weapon)
        self.armor = armor  # Ny attributt for rustning med default verdi 5

    def take_damage(self, amount: int) -> None:
        """Reduces incoming damage by armor before applying it (minimum 0 damage)[cite: 3]"""
        reduced_damage = max(amount - self.armor, 0)
        super().take_damage(reduced_damage)

    def __repr__(self) -> str:
        return f"<Warrior name={self.name}, hp={self.hp}, armor={self.armor}, weapon={self.weapon}>"


class Mage(Character):
    """Represents a resource-based attacker Mage character[cite: 3]"""

    def __init__(self, name: str, hp_max: int = 100, damage: int = 10, weapon: Weapon = None, mana: int = 30, mana_max: int = 30) -> None:
        super().__init__(name, hp_max, damage, weapon)
        self.mana = mana          # Ny attributt for nåværende mana (default 30)[cite: 3]
        self.mana_max = mana_max  # Ny attributt for maks mana (default 30)[cite: 3]

    def attack(self, target) -> None:
        """Attacks target using magic if enough mana, otherwise falls back to base weapon attack and restores mana[cite: 3]"""
        if self.mana >= 10:
            self.mana -= 10  # Bruker 10 mana[cite: 3]
            inflicted_damage = self.damage * 2  # Magisk angrep gir dobbel skade og bruker ikke våpen[cite: 3]
            target.take_damage(inflicted_damage)
        else:
            # Hvis mindre enn 10 mana: gjenopprett 5 mana og bruk standard/våpen-angrep[cite: 3]
            self.mana = min(self.mana + 5, self.mana_max)
            super().attack(target)

    def __repr__(self) -> str:
        return f"<Mage name={self.name}, hp={self.hp}, mana={self.mana}/{self.mana_max}, weapon={self.weapon}>"

In [3]:
""" ----- Task 2 ----- Dice class -----"""

import random as rd

class Dice:
    """Represents the dice that will be used in the game"""


    def __init__(self, sides: int = 6):
        """Initialize dice
        Args:
            sides(int):        Amount of sides on dice, default value = 6 (standard dice)"""
        if sides < 2:
            raise ValueError("Minimum amount of sides on dice = 2")
        
        self.sides = sides
        


    def roll(self) -> int:
        """Roll the dice! Returns a random number between 1 and the total amount of sides on the dice"""
        return rd.randint(1, self.sides)


    def __repr__(self)->str:
        """Representation of dice object"""
        return f"Dice(sides = {self.sides})"


    def __call__(self) -> int:
        """Allows the object to be called as function"""
        return self.roll()

d20 = Dice(20)
d6 = Dice()
print(d6, ',', d20)
print(d6(),',',d20())



Dice(sides = 6) , Dice(sides = 20)
5 , 14


In [4]:
""" ----- Task 3 ----- Create items -----"""

from dice import Dice

class Weapon:
    """Represents a weapon in the game"""


    def __init__(self, name:str, damage_dice: Dice)->None:
        """Initialize weapon"""
        self.name = name
        self.damage_dice = damage_dice


    def roll_damage(self) ->int:
        """Dice that rolls damage points given to target"""
        return self.damage_dice()


    def __repr__(self) -> None:
        """Representation of weapon object"""
        return f"Weapon(name = '{self.name}', damage_dice = {self.damage_dice})"


    def __str__(self) -> None:
        """String representation of weapon object"""
        return f"Weapon: {self.name}, (Dice: {self.damage_dice})"
    



In [7]:
""" ----- Task 5 ----- run battles ----- """

from character import Character, Warrior, Mage

class Battle:
    """Manages turn-based combat between two Character objects"""

    def __init__(self, hero: Character, enemy: Character) -> None:
        self.hero = hero
        self.enemy = enemy

    def play_round(self) -> None:
        """Executes a single combat round where hero attacks first, then enemy if still alive"""
        if self.hero.is_alive:
            self.hero.attack(self.enemy)
            print(f"{self.hero.name} angriper {self.enemy.name}! {self.enemy.name} har nå {self.enemy.hp} HP igjen.")

        if self.enemy.is_alive:
            self.enemy.attack(self.hero)
            print(f"{self.enemy.name} angriper {self.hero.name}! {self.hero.name} har nå {self.hero.hp} HP igjen.")

    def simulate_battle(self) -> Character:
        """Simulates turns in a while loop until one character is defeated, returns the winner"""
        print(f"--- Kampen starter mellom {self.hero.name} og {self.enemy.name}! ---")
        
        while self.hero.is_alive and self.enemy.is_alive:
            self.play_round()
            print("-" * 30)

        winner = self.hero if self.hero.is_alive else self.enemy
        print(f"🏆 {winner.name} har vunnet kampen!")
        return winner

    # --- SIMULERT KAMP FOR INNLEVERINGEN ---

# 1. Lag en terning og et våpen (Task 2 & 3)
sword_dice = Dice(8)
iron_sword = Weapon(name="Iron Sword", damage_dice=sword_dice)

# 2. Opprett en Warrior og en Mage (Task 1 & 4)
# Krigeren får et sverd og standard rustning (armor=5)
hero_warrior = Warrior(name="Johnny", hp_max=100, damage=10, weapon=iron_sword, armor=5)

# Magen starter med 30 mana og magiske angrep
enemy_mage = Mage(name="Adam_smasher", hp_max=80, damage=12, mana=30)

# 3. Start turbasert kamp (Task 5)
print(hero_warrior)
print(enemy_mage)
print("=" * 40)

epic_battle = Battle(hero=hero_warrior, enemy=enemy_mage)
winner = epic_battle.simulate_battle()


Character: Johnny with 100 HP and 10 damage  with Weapon: Iron Sword, (Dice: Dice(sides = 8))
Character: Adam_smasher with 80 HP and 12 damage 
--- Kampen starter mellom Johnny og Adam_smasher! ---
Johnny angriper Adam_smasher! Adam_smasher har nå 74 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 81 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 66 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 62 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 58 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 43 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 52 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 36 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 47 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 29 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har 

In [13]:
"""Battle 2"""

knife_dice = Dice(4)
butter_knife = Weapon(name="Butter knife", damage_dice=knife_dice)

# 2. Opprett en Warrior og en Mage (Task 1 & 4)
# Krigeren får en kniv og tung rustning (armor=20)
hero_warrior = Warrior(name="Johnny", hp_max=100, damage=10, weapon=butter_knife, armor=20)

# Magen starter med 30 mana og magiske angrep
enemy_mage = Mage(name="Adam_smasher", hp_max=80, damage=12, mana=30)

# 3. Start turbasert kamp (Task 5)
print(hero_warrior)
print(enemy_mage)
print("=" * 40)

epic_battle = Battle(hero=hero_warrior, enemy=enemy_mage)
winner = epic_battle.simulate_battle()

Character: Johnny with 100 HP and 10 damage  with Weapon: Butter knife, (Dice: Dice(sides = 4))
Character: Adam_smasher with 80 HP and 12 damage 
--- Kampen starter mellom Johnny og Adam_smasher! ---
Johnny angriper Adam_smasher! Adam_smasher har nå 78 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 96 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 76 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 92 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 75 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 73 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 72 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher ha

In [14]:
""" Battle 3"""
"""Fighting with bare hands"""
fist_dice = Dice(3)
butter_knife = Weapon(name="Bare hands", damage_dice=fist_dice)

# 2. Opprett en Warrior og en Mage (Task 1 & 4)
# Krigeren får en kniv og tung rustning (armor=20)
hero_warrior = Warrior(name="Johnny", hp_max=100, damage=10, weapon=butter_knife, armor=20)

# Magen starter med 30 mana og magiske angrep
enemy_mage = Mage(name="Adam_smasher", hp_max=80, damage=12, mana=30)

# 3. Start turbasert kamp (Task 5)
print(hero_warrior)
print(enemy_mage)
print("=" * 40)

epic_battle = Battle(hero=hero_warrior, enemy=enemy_mage)
winner = epic_battle.simulate_battle()

Character: Johnny with 100 HP and 10 damage  with Weapon: Bare hands, (Dice: Dice(sides = 3))
Character: Adam_smasher with 80 HP and 12 damage 
--- Kampen starter mellom Johnny og Adam_smasher! ---
Johnny angriper Adam_smasher! Adam_smasher har nå 78 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 96 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 76 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 92 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 73 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 71 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har nå 69 HP igjen.
Adam_smasher angriper Johnny! Johnny har nå 88 HP igjen.
------------------------------
Johnny angriper Adam_smasher! Adam_smasher har 